# Imports + class

In [43]:
import xgboost as xgb
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np
from z3 import *
from xgboost import XGBClassifier
from pmlb import fetch_data

set_option(rational_to_decimal=True)

import re
import time
import pandas as pd
from sklearn.model_selection import train_test_split

np.set_printoptions(suppress=True)

In [44]:
import math


def sigmoid(x):
    return 1 / (1 + math.exp(-x))

In [45]:
class XGBoostExplainer:
    def __init__(self, model, data):
        self.model = model
        self.data = data.values
        self.columns = data.columns
        self.max_categories = 2

        set_option(rational_to_decimal=True)
        self.categoric_features = self.get_categoric_features(self.data)
        self.T_model = self.model_trees_expression(self.model)
        self.T = self.T_model

    def explain(self, instance, reorder="asc"):
        self.I = self.instance_expression(instance)
        self.D, self.D_add = self.decision_function_expression(self.model, [instance])
        return self.explain_expression(
            self.I, And(self.T, self.D_add), self.D, self.model, reorder
        )

    def explain_prob(
        self, instance, reorder="asc", threshold_margin=0, target_threshold=None
    ):
        try:
            self.I = self.instance_expression(instance)
            self.D, self.D_add = self.decision_function_expression(
                self.model, [instance]
            )
            return self.explain_expression_prob(
                self.I,
                And(self.T, self.D_add),
                self.D,
                self.model,
                reorder,
                threshold_margin,
                target_threshold,
            )
        except Exception as e:
            print("Error in explain_prob:", e)
            return []

    def get_categoric_features(self, data: np.ndarray):
        categoric_features = []
        for i in range(data.shape[1]):
            feature_values = data[:, i]
            unique_values = np.unique(feature_values)
            if len(unique_values) <= self.max_categories:
                categoric_features.append(self.columns[i])

        return categoric_features

    def feature_constraints(self, constraints=[]):
        """TODO
        esperado receber limites das features pelo usuário
        formato previso: matriz/dataframe [feaature, min/max, valor]
        constraaint_expression = "constraaint_df_to_feature()"
        """
        return

    def model_trees_expression(self, model):
        df = model.get_booster().trees_to_dataframe()
        if model.get_booster().feature_names == None:
            feature_map = {f"f{i}": col for i, col in enumerate(self.columns)}
            df["Feature"] = df["Feature"].replace(feature_map)

        df["Split"] = df["Split"].round(4)
        self.booster_df = df
        class_index = 0  # if model.n_classes_ == 2:
        all_tree_formulas = []

        for tree_index in df["Tree"].unique():
            tree_df = df[df["Tree"] == tree_index]
            o = Real(f"o_{tree_index}_{class_index}")

            if len(tree_df) == 1 and tree_df.iloc[0]["Feature"] == "Leaf":
                leaf_value = tree_df.iloc[0]["Gain"]
                all_tree_formulas.append(And(o == leaf_value))
                continue
            path_formulas = []

            def get_conditions(node_id):
                conditions = []
                current_node = tree_df[tree_df["ID"] == node_id]
                if current_node.empty:
                    return conditions

                parent_node = tree_df[
                    (tree_df["Yes"] == node_id) | (tree_df["No"] == node_id)
                ]
                if not parent_node.empty:
                    parent_data = parent_node.iloc[0]
                    feature = parent_data["Feature"]
                    split_value = parent_data["Split"]
                    x = Real(feature)
                    if parent_data["Yes"] == node_id:
                        conditions.append(x < split_value)
                    else:
                        conditions.append(x >= split_value)
                    conditions = get_conditions(parent_data["ID"]) + conditions

                return conditions

            for _, node in tree_df[tree_df["Feature"] == "Leaf"].iterrows():
                leaf_value = node["Gain"]
                leaf_id = node["ID"]
                conditions = get_conditions(leaf_id)
                path_formula = And(*conditions)
                implication = Implies(path_formula, o == leaf_value)
                path_formulas.append(implication)

            all_tree_formulas.append(And(*path_formulas))
        return And(*all_tree_formulas)

    def get_init_value(self, model, x, estimator_variables):
        estimator_pred = Solver()
        estimator_pred.add(self.I)
        estimator_pred.add(self.T)
        if estimator_pred.check() == sat:
            solvermodel = estimator_pred.model()
            total_sum = sum(
                float(solvermodel.eval(var).as_fraction())
                for var in estimator_variables
            )
        else:
            total_sum = 0
            print("estimator error")
        self.predicted_margin = model.predict(x, output_margin=True)[0]
        init_value = self.predicted_margin - total_sum
        self.init_value = init_value
        return init_value

    def decision_function_expression(self, model, x):
        n_classes = 1 if model.n_classes_ <= 2 else model.n_classes_
        predicted_class = model.predict(x)[0]
        self.predicted_class = predicted_class
        n_estimators = int(len(model.get_booster().get_dump()) / n_classes)
        estimator_variables = [
            Real(f"o_{i}_0") for i in range(n_estimators)
        ]  # _0 only for binary classification
        self.estimator_variables = estimator_variables
        init_value = self.get_init_value(model, x, estimator_variables)
        # print("init:", round(init_value, 2))

        equation_list = []

        estimator_sum = Real("estimator_sum")
        equation_o = estimator_sum == Sum(estimator_variables)
        equation_list.append(equation_o)

        decision = Real("decision")
        equation_list.append(decision == estimator_sum + init_value)

        if predicted_class == 0:
            final_equation = decision < 0
        else:
            final_equation = decision > 0

        return final_equation, And(equation_list)

    def instance_expression(self, instance):
        formula = [Real(self.columns[i]) == value for i, value in enumerate(instance)]
        return formula

    def explain_expression(self, I, T_s, D_s, model, reorder):
        i_expression = I.copy()

        importances = model.feature_importances_
        non_zero_indices = np.where(importances != 0)[0]

        if reorder == "asc":
            sorted_feature_indices = non_zero_indices[
                np.argsort(importances[non_zero_indices])
            ]
            i_expression = [i_expression[i] for i in sorted_feature_indices]
        elif reorder == "desc":
            sorted_feature_indices = non_zero_indices[
                np.argsort(-importances[non_zero_indices])
            ]
            i_expression = [i_expression[i] for i in sorted_feature_indices]

        for feature in i_expression.copy():
            # print("\n---removed", feature)
            i_expression.remove(feature)

            # prove(Implies(And(And(i_expression), T), D))
            if self.is_proved(Implies(And(And(i_expression), T_s), D_s)):
                continue
                # print('proved')
            else:
                # print('not proved')
                i_expression.append(feature)
        # print(self.is_proved(Implies(And(And(i_expression), T_s), D_s)))
        return i_expression

    def explain_expression_prob(
        self, I, T_s, D_s, model, reorder, threshold_margin, target_threshold
    ):
        i_expression = I.copy()

        importances = model.feature_importances_
        non_zero_indices = np.where(importances != 0)[0]

        if reorder == "asc":
            sorted_feature_indices = non_zero_indices[
                np.argsort(importances[non_zero_indices])
            ]
            i_expression = [i_expression[i] for i in sorted_feature_indices]
        elif reorder == "desc":
            sorted_feature_indices = non_zero_indices[
                np.argsort(-importances[non_zero_indices])
            ]
            i_expression = [i_expression[i] for i in sorted_feature_indices]

        threshold = 0

        if target_threshold:
            threshold = target_threshold
        elif threshold_margin != 0:
            threshold = self.predicted_margin * threshold_margin / 100
            # print("margin:", self.predicted_margin, "accepted margin:", threshold)
        self.xai_predicted_margin = self.predicted_margin

        for feature in i_expression.copy():
            # print("\n---removed", feature)
            i_expression.remove(feature)

            if self.is_proved_sat(And(And(i_expression), T_s), threshold):
                # print('proved')
                continue
            else:
                # print('not proved -- added back')
                i_expression.append(feature)
        return i_expression

    def is_proved(self, decision_exp):
        s = Solver()
        s.add(Not(decision_exp))
        if s.check() == unsat:
            return True
        else:
            # print(s.model())
            return False

    def is_proved_sat(self, decision_exp, threshold):
        decision = Real("decision")

        debug = Real("debug") == 0
        predicted_class = self.predicted_class

        if predicted_class == 0:
            estmax = Optimize()
            estmax.add(decision_exp)
            estmax.add(debug)
            maxvalue = estmax.maximize(decision)
            if estmax.check() == sat:
                # print("\nmax sat", maxvalue.value())
                try:
                    if float(maxvalue.value().as_fraction()) > threshold:
                        return False  # can change class
                    else:
                        self.xai_predicted_margin = float(
                            maxvalue.value().as_fraction()
                        )
                except:
                    print("error max =", maxvalue.value())
                    return False
            else:
                print("error")

        if predicted_class == 1:
            estmin = Optimize()
            estmin.add(decision_exp)
            estmin.add(debug)
            minvalue = estmin.minimize(decision)
            if estmin.check() == sat:
                # print("\nmin sat", minvalue.value())
                try:
                    if float(minvalue.value().as_fraction()) < threshold:
                        return False  # can change class
                    else:
                        self.xai_predicted_margin = float(
                            minvalue.value().as_fraction()
                        )
                except:
                    print("error min =", minvalue.value())
                    return False
            else:
                print("error")

        if predicted_class == 0:
            self.solvermodel = estmax.model()
        if predicted_class == 1:
            self.solvermodel = estmin.model()
        return True


def generate_results(explainer, X_test, y_pred, classification, path, reorder="asc"):
    results = []
    if classification == 0:
        increase_prob = -0.01
    else:
        increase_prob = 0.01
    for i in X_test[y_pred == classification].index:
        sample = X_test.loc[i].values
        xai = explainer.explain_prob(sample, reorder=reorder)
        xaiprob_initial = explainer.xai_predicted_margin
        len_xai_initial = len(xai)

        xai = explainer.explain_prob(
            sample, reorder=reorder, target_threshold=xaiprob_initial + increase_prob
        )
        xaiprob_final = explainer.xai_predicted_margin
        len_xai_final = len(xai)

        if round(xaiprob_initial, 2) == round(xaiprob_final, 2):
            len_xai_initial = len_xai_final
        results.append(
            {
                "index": i,
                "class": classification,
                "xaiprob_initial": round(xaiprob_initial, 2),
                "len_xai_initial": len_xai_initial,
                "xaiprob_final": round(xaiprob_final, 2),
                "len_xai_final": len_xai_final,
            }
        )
    df_results = pd.DataFrame(results)
    return df_results
    # df_results.to_csv(f'{path}/results_{classification}_{reorder}.csv', index=False)

# Prepare datasets and models

In [46]:
def prepare_dataset_model_explainer(dataset_name, dataset_params):
    # Fetch data
    dataset = fetch_data(dataset_name)
    X = dataset.drop("target", axis=1)
    y = dataset["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    params = dataset_params[dataset_name]

    # Train model
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Metrics
    print("Dataset size:", len(X))
    print("Columns:", len(X.columns))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))

    # Explainer
    explainer = XGBoostExplainer(model, X_train)

    return model, X, y, X_test, y_test, explainer


def prepare_all_datasets(dataset_names, dataset_params):
    context = {}
    for name in dataset_names:
        print(f"\n--- Preparing {name} ---")
        model, X, y, X_test, y_test, explainer = prepare_dataset_model_explainer(
            name, dataset_params
        )
        context[name] = (model, X, y, X_test, y_test, explainer)
    return context

In [47]:
dataset_names = [
    "magic",
    "adult",
    "mushroom",
    "spambase",
]

dataset_params = {
    "magic": {"n_estimators": 30, "max_depth": 3},
    "adult": {"n_estimators": 30, "max_depth": 3},
    "mushroom": {"n_estimators": 30, "max_depth": 3},
    "spambase": {"n_estimators": 30, "max_depth": 3},
}

dataset_context = prepare_all_datasets(dataset_names, dataset_params)


--- Preparing magic ---
Dataset size: 19020
Columns: 10
Accuracy: 0.8755695758850333
F1-score: 0.8060109289617486

--- Preparing adult ---
Dataset size: 48842
Columns: 14
Accuracy: 0.8652835596806114
F1-score: 0.9144491635607177

--- Preparing mushroom ---
Dataset size: 8124
Columns: 22
Accuracy: 1.0
F1-score: 1.0

--- Preparing spambase ---
Dataset size: 4601
Columns: 57
Accuracy: 0.9471397538015931
F1-score: 0.9322191272051996


In [48]:
def count_classes(dataset_name):
    model, X, y, X_test, y_test, explainer = dataset_context[dataset_name]
    y_pred = model.predict(X_test)
    class_0_count = np.sum(y_pred == 0)
    class_1_count = np.sum(y_pred == 1)
    print(
        f"Dataset: {dataset_name}, Class 0: {class_0_count}, Class 1: {class_1_count}"
    )
    return class_0_count, class_1_count


count_classes("magic")
count_classes("adult")
count_classes("mushroom")
count_classes("spambase")

Dataset: magic, Class 0: 4052, Class 1: 1654
Dataset: adult, Class 0: 2726, Class 1: 11927
Dataset: mushroom, Class 0: 1263, Class 1: 1175
Dataset: spambase, Class 0: 848, Class 1: 533


(np.int64(848), np.int64(533))

# Check explanations

In [49]:
def get_explain_results(dataset_name, print_exp=False):
    model, X, y, X_test, y_test, explainer = dataset_context[dataset_name]
    y_pred = model.predict(X_test)

    # Function to explain a sample of a given class
    def explain_for_class(classification):
        # Get a sample of the specified class in predictions
        indices = X_test[y_pred == classification].index
        if len(indices) == 0:
            print(f"No samples predicted as class {classification}.")
            return

        for i in indices:
            sample = X_test.loc[i].values
            margin = model.predict([sample], output_margin=True)[0]
            if abs(margin) > 1.0:
                break
        else:
            print(f"No samples with |margin| > 1.0 for class {classification}.")
            return

        # Initial explanation
        margin = model.predict([sample], output_margin=True)[0]
        print(f"\nClass {classification} - Margin: {margin}, Sample index: {i}")

        xai = explainer.explain_prob(sample)
        if print_exp:
            print("Initial abductive explanation:", xai)

        xaiprob_initial = explainer.xai_predicted_margin
        print("Initial exp len:", len(xai))
        print("Initial predicted margin:", xaiprob_initial)

        # Adjust margin direction
        increase_prob = -0.01 if classification == 0 else 0.01

        xai_confidence = explainer.explain_prob(
            sample, target_threshold=(xaiprob_initial + increase_prob)
        )
        xaiprob_final = explainer.xai_predicted_margin
        print("\nincreased exp len:", len(xai_confidence))
        if print_exp:
            print("increased explanation:", xai_confidence)
        print("increased predicted margin:", xaiprob_final)

        # Explanation with adjusted threshold
        xai_confidence = explainer.explain_prob(sample, target_threshold=(margin / 4))
        xaiprob_final = explainer.xai_predicted_margin
        print("\n0.25% exp len:", len(xai_confidence))
        if print_exp:
            print("0.25% explanation:", xai_confidence)
        print("0.25% predicted margin:", xaiprob_final)

        xai_confidence = explainer.explain_prob(sample, target_threshold=(margin / 2))
        xaiprob_final = explainer.xai_predicted_margin
        print("\n0.5% exp len:", len(xai_confidence))
        if print_exp:
            print("0.5% explanation:", xai_confidence)
        print("0.5% predicted margin:", xaiprob_final)

        xai_confidence = explainer.explain_prob(
            sample, target_threshold=(margin / 1.33333)
        )
        xaiprob_final = explainer.xai_predicted_margin
        print("\n0.75% exp len:", len(xai_confidence))
        if print_exp:
            print("0.75% explanation:", xai_confidence)
        print("0.75% predicted margin:", xaiprob_final)

    # Explain for both classes
    explain_for_class(0)
    explain_for_class(1)

In [50]:
get_explain_results("magic")


Class 0 - Margin: -1.5318019390106201, Sample index: 13519
Initial exp len: 7
Initial predicted margin: -0.305143745439

increased exp len: 8
increased predicted margin: -0.842865952239

0.25% exp len: 8
0.25% predicted margin: -0.842865952239

0.5% exp len: 8
0.5% predicted margin: -0.842865952239

0.75% exp len: 9
0.75% predicted margin: -1.438409154139

Class 1 - Margin: 5.3488240242004395, Sample index: 14526
Initial exp len: 3
Initial predicted margin: 0.298496220521

increased exp len: 4
increased predicted margin: 0.451575534982

0.25% exp len: 3
0.25% predicted margin: 3.056817650122

0.5% exp len: 3
0.5% predicted margin: 3.056817650122

0.75% exp len: 4
0.75% predicted margin: 4.514383172382


In [51]:
get_explain_results("adult", print_exp=True)


Class 0 - Margin: -1.0879830121994019, Sample index: 3829
Initial abductive explanation: [capital-loss == 0, occupation == 4, age == 40, hours-per-week == 46, capital-gain == 0, education-num == 14, relationship == 0]
Initial exp len: 7
Initial predicted margin: -0.201337004746

increased exp len: 8
increased explanation: [capital-loss == 0, occupation == 4, age == 40, hours-per-week == 46, capital-gain == 0, marital-status == 2, education-num == 14, relationship == 0]
increased predicted margin: -0.623709311386

0.25% exp len: 8
0.25% explanation: [capital-loss == 0, occupation == 4, age == 40, hours-per-week == 46, capital-gain == 0, marital-status == 2, education-num == 14, relationship == 0]
0.25% predicted margin: -0.623709311386

0.5% exp len: 8
0.5% explanation: [capital-loss == 0, occupation == 4, age == 40, hours-per-week == 46, capital-gain == 0, marital-status == 2, education-num == 14, relationship == 0]
0.5% predicted margin: -0.623709311386

0.75% exp len: 10
0.75% expla

In [52]:
get_explain_results("mushroom")


Class 0 - Margin: -7.027366638183594, Sample index: 7176
Initial exp len: 3
Initial predicted margin: -4.6045556678

increased exp len: 4
increased predicted margin: -5.0068300378

0.25% exp len: 3
0.25% predicted margin: -4.6045556678

0.5% exp len: 3
0.5% predicted margin: -4.6045556678

0.75% exp len: 5
0.75% predicted margin: -5.6530501859

Class 1 - Margin: 8.294289588928223, Sample index: 4961
Initial exp len: 2
Initial predicted margin: 0.0284762922

increased exp len: 3
increased predicted margin: 2.0411129242

0.25% exp len: 4
0.25% predicted margin: 3.9462458302

0.5% exp len: 5
0.5% predicted margin: 5.1567087872

0.75% exp len: 7
0.75% predicted margin: 6.4058943492


In [53]:
get_explain_results("spambase")


Class 0 - Margin: -6.460454940795898, Sample index: 4538
Initial exp len: 10
Initial predicted margin: -0.04732684392

increased exp len: 11
increased predicted margin: -0.36002053017

0.25% exp len: 13
0.25% predicted margin: -1.68991961052

0.5% exp len: 17
0.5% predicted margin: -3.317425186

0.75% exp len: 19
0.75% predicted margin: -4.8820868495

Class 1 - Margin: 4.380174160003662, Sample index: 1685
Initial exp len: 14
Initial predicted margin: 0.1421553674

increased exp len: 14
increased predicted margin: 0.1551693114

0.25% exp len: 15
0.25% predicted margin: 1.1584000099

0.5% exp len: 18
0.5% predicted margin: 2.21234623721

0.75% exp len: 21
0.75% predicted margin: 3.2869784274


# Check threshold datasets

## functions

In [54]:
# def get_explain_results_n_increased(dataset_name, n_samples):
#     model, X, y, X_test, y_test, explainer = dataset_context[dataset_name]
#     y_pred = model.predict(X_test)

#     # Prepare result DataFrame
#     results = []
#     # Loop over both classes
#     for classification in [0, 1]:
#         # Get indices for predicted samples of this class
#         indices = X_test[y_pred == classification].index[:n_samples]
#         if len(indices) == 0:
#             print(f"No samples predicted as class {classification}.")
#             continue

#         for idx in indices:
#             sample = X_test.loc[idx].values

#             # Initial explanation
#             margin = model.predict([sample], output_margin=True)[0]
#             xai_initial = explainer.explain_prob(sample)
#             xaiprob_initial = sigmoid(explainer.xai_predicted_margin)
#             exp_len_initial = len(xai_initial)

#             increase_prob = -0.01 if margin < 0 else 0.01
#             xai_confidence_increased = explainer.explain_prob(
#                 sample, target_threshold=xaiprob_initial + increase_prob
#             )
#             xaiprob_increased = explainer.xai_predicted_margin
#             exp_len_increased = len(xai_confidence_increased)

#             same_len = exp_len_initial == exp_len_increased
#             # Append to results
#             results.append(
#                 {
#                     "sample_id": idx,
#                     "class": classification,
#                     "pred_margin": round(margin, 4),
#                     "exp_len_inicial": exp_len_initial,
#                     "xaiprob_inicial": round(xaiprob_initial, 4),
#                     "exp_len_increased": exp_len_increased,
#                     "xaiprob_increased": round(xaiprob_increased, 4),
#                     "same_len_count": same_len,
#                 }
#             )

#     # Convert to DataFrame
#     df_results = pd.DataFrame(results)

#     return df_results


# def summarize_explanations_increased(df_results):
#     # Agrupar por classe e calcular as métricas
#     summary = (
#         df_results.groupby("class")
#         .agg(
#             pred_margin_mean=("pred_margin", "mean"),
#             pred_margin_std=("pred_margin", "std"),
#             exp_len_inicial_mean=("exp_len_inicial", "mean"),
#             exp_len_inicial_std=("exp_len_inicial", "std"),
#             xaiprob_inicial_mean=("xaiprob_inicial", "mean"),
#             xaiprob_inicial_std=("xaiprob_inicial", "std"),
#             exp_len_increased_mean=("exp_len_increased", "mean"),
#             exp_len_increased_std=("exp_len_increased", "std"),
#             xaiprob_increased_mean=("xaiprob_increased", "mean"),
#             xaiprob_increased_std=("xaiprob_increased", "std"),
#         )
#         .reset_index()
#     )

#     # Função para combinar mean ± std com 2 casas decimais
#     def format_mean_std(mean, std):
#         return f"{mean:.2f} ± {std:.2f}"

#     # Aplicar a formatação em cada par de colunas
#     formatted = pd.DataFrame()
#     formatted["class"] = summary["class"]
#     formatted["Pred Margin"] = summary.apply(
#         lambda row: format_mean_std(row["pred_margin_mean"], row["pred_margin_std"]),
#         axis=1,
#     )
#     formatted["Exp Len Inicial"] = summary.apply(
#         lambda row: format_mean_std(
#             row["exp_len_inicial_mean"], row["exp_len_inicial_std"]
#         ),
#         axis=1,
#     )
#     formatted["MCT Inicial"] = summary.apply(
#         lambda row: format_mean_std(
#             row["xaiprob_inicial_mean"], row["xaiprob_inicial_std"]
#         ),
#         axis=1,
#     )
#     formatted["Exp Len increased"] = summary.apply(
#         lambda row: format_mean_std(row["exp_len_increased_mean"], row["exp_len_increased_std"]),
#         axis=1,
#     )
#     formatted["MCT increased"] = summary.apply(
#         lambda row: format_mean_std(row["xaiprob_increased_mean"], row["xaiprob_increased_std"]),
#         axis=1,
#     )
#     formatted["Same Len Count"] = df_results.groupby("class")["same_len_count"].sum().values

#     return formatted

In [58]:
def get_explain_results_n(dataset_name, n_samples):
    model, X, y, X_test, y_test, explainer = dataset_context[dataset_name]
    y_pred = model.predict(X_test)

    # Prepare result DataFrame
    results = []

    # Loop over both classes
    for classification in [0, 1]:
        # Get indices for predicted samples of this class
        indices = X_test[y_pred == classification].index[:n_samples]
        if len(indices) == 0:
            print(f"No samples predicted as class {classification}.")
            continue

        same_len_count = 0
        for idx in indices:
            sample = X_test.loc[idx].values

            # Initial explanation
            margin = model.predict([sample], output_margin=True)[0]
            margin_sigmoid = sigmoid(margin)

            xai_initial = explainer.explain_prob(sample)
            xai_margin_initial = explainer.xai_predicted_margin
            xaiprob_initial = sigmoid(explainer.xai_predicted_margin)
            exp_len_initial = len(xai_initial)

            increase_prob = -0.01 if margin < 0 else 0.01
            xai_confidence_increased = explainer.explain_prob(
                sample, target_threshold = (xai_margin_initial + increase_prob)
            )
            xaiprob_increased = sigmoid(explainer.xai_predicted_margin)
            exp_len_increased = len(xai_confidence_increased)

            if exp_len_initial == exp_len_increased:
                same_len_count += 1

            # Explanation with adjusted threshold (half the margin)
            xai_confidence_25 = explainer.explain_prob(
                sample, target_threshold=margin / 4
            )
            xaiprob_25 = sigmoid(explainer.xai_predicted_margin)
            exp_len_25 = len(xai_confidence_25)

            # Explanation with adjusted threshold (half the margin)
            xai_confidence_50 = explainer.explain_prob(
                sample, target_threshold=margin / 2
            )
            xaiprob_50 = sigmoid(explainer.xai_predicted_margin)
            exp_len_50 = len(xai_confidence_50)

            # Explanation with adjusted threshold (half the margin)
            xai_confidence_75 = explainer.explain_prob(
                sample, target_threshold=margin / 1.3333
            )
            xaiprob_75 = sigmoid(explainer.xai_predicted_margin)
            exp_len_75 = len(xai_confidence_75)

            # Append to results
            results.append(
                {
                    "sample_id": idx,
                    "class": classification,
                    "pred_margin": round(margin_sigmoid, 4),
                    "exp_len_inicial": exp_len_initial,
                    "xaiprob_inicial": round(xaiprob_initial, 4),
                    "exp_len_increased": exp_len_increased,
                    "xaiprob_increased": round(xaiprob_increased, 4),
                    "exp_len_25": exp_len_25,
                    "xaiprob_25": round(xaiprob_25, 4),
                    "exp_len_50": exp_len_50,
                    "xaiprob_50": round(xaiprob_50, 4),
                    "exp_len_75": exp_len_75,
                    "xaiprob_75": round(xaiprob_75, 4),
                }
            )
        print(classification, "same len on increase:", same_len_count)

    # Convert to DataFrame
    df_results = pd.DataFrame(results)

    return df_results


def summarize_explanations(df_results):
    # Agrupar por classe e calcular as métricas
    summary = (
        df_results.groupby("class")
        .agg(
            pred_margin_mean=("pred_margin", "mean"),
            pred_margin_std=("pred_margin", "std"),
            exp_len_inicial_mean=("exp_len_inicial", "mean"),
            exp_len_inicial_std=("exp_len_inicial", "std"),
            xaiprob_inicial_mean=("xaiprob_inicial", "mean"),
            xaiprob_inicial_std=("xaiprob_inicial", "std"),
            exp_len_increased_mean=("exp_len_increased", "mean"),
            exp_len_increased_std=("exp_len_increased", "std"),
            xaiprob_increased_mean=("xaiprob_increased", "mean"),
            xaiprob_increased_std=("xaiprob_increased", "std"),
            exp_len_25_mean=("exp_len_25", "mean"),
            exp_len_25_std=("exp_len_25", "std"),
            xaiprob_25_mean=("xaiprob_25", "mean"),
            xaiprob_25_std=("xaiprob_25", "std"),
            exp_len_50_mean=("exp_len_50", "mean"),
            exp_len_50_std=("exp_len_50", "std"),
            xaiprob_50_mean=("xaiprob_50", "mean"),
            xaiprob_50_std=("xaiprob_50", "std"),
            exp_len_75_mean=("exp_len_75", "mean"),
            exp_len_75_std=("exp_len_75", "std"),
            xaiprob_75_mean=("xaiprob_75", "mean"),
            xaiprob_75_std=("xaiprob_75", "std"),
        )
        .reset_index()
    )

    # Função para combinar mean ± std com 2 casas decimais
    def format_mean_std(mean, std):
        return f"{mean:.2f} ± {std:.2f}"

    # Aplicar a formatação em cada par de colunas
    formatted = pd.DataFrame()
    formatted["class"] = summary["class"]
    formatted["Pred Margin"] = summary.apply(
        lambda row: format_mean_std(row["pred_margin_mean"], row["pred_margin_std"]),
        axis=1,
    )
    formatted["Exp Len Inicial"] = summary.apply(
        lambda row: format_mean_std(
            row["exp_len_inicial_mean"], row["exp_len_inicial_std"]
        ),
        axis=1,
    )
    formatted["MCT Inicial"] = summary.apply(
        lambda row: format_mean_std(
            row["xaiprob_inicial_mean"], row["xaiprob_inicial_std"]
        ),
        axis=1,
    )
    formatted["Exp Len increased"] = summary.apply(
        lambda row: format_mean_std(row["exp_len_increased_mean"], row["exp_len_increased_std"]),
        axis=1,
    )
    formatted["MCT increased"] = summary.apply(
        lambda row: format_mean_std(row["xaiprob_increased_mean"], row["xaiprob_increased_std"]),
        axis=1,
    )
    formatted["Exp Len 25"] = summary.apply(
        lambda row: format_mean_std(row["exp_len_25_mean"], row["exp_len_25_std"]),
        axis=1,
    )
    formatted["MCT 25"] = summary.apply(
        lambda row: format_mean_std(row["xaiprob_25_mean"], row["xaiprob_25_std"]),
        axis=1,
    )
    formatted["Exp Len 50"] = summary.apply(
        lambda row: format_mean_std(row["exp_len_50_mean"], row["exp_len_50_std"]),
        axis=1,
    )
    formatted["MCT 50"] = summary.apply(
        lambda row: format_mean_std(row["xaiprob_50_mean"], row["xaiprob_50_std"]),
        axis=1,
    )
    formatted["Exp Len 75"] = summary.apply(
        lambda row: format_mean_std(row["exp_len_75_mean"], row["exp_len_75_std"]),
        axis=1,
    )
    formatted["MCT 75"] = summary.apply(
        lambda row: format_mean_std(row["xaiprob_75_mean"], row["xaiprob_75_std"]),
        axis=1,
    )

    return formatted

In [ ]:
# run up

# datasets increased

In [86]:
instances_to_explain = 100

In [87]:
df_saheart = get_explain_results_n_increased("magic", instances_to_explain)
df_summary_saheart = summarize_explanations_increased(df_saheart)
df_summary_saheart

,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Same Len Count
0,0,-2.00 ± 1.03,6.48 ± 1.01,0.44 ± 0.06,5.77 ± 0.92,0.11 ± 0.34,32
1,1,2.32 ± 1.61,3.52 ± 1.12,0.59 ± 0.09,4.75 ± 2.35,0.83 ± 0.41,33


In [88]:
df_adult = get_explain_results_n_increased("adult", instances_to_explain)
df_summary_adult = summarize_explanations_increased(df_adult)
df_summary_adult

,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Same Len Count
0,0,-1.79 ± 1.64,7.35 ± 2.46,0.45 ± 0.04,6.33 ± 2.08,0.23 ± 0.17,21
1,1,2.65 ± 1.60,4.86 ± 1.02,0.58 ± 0.07,5.88 ± 2.80,0.86 ± 0.37,22


In [89]:
df_mushroom = get_explain_results_n_increased("mushroom", instances_to_explain)
df_summary_mushroom = summarize_explanations_increased(df_mushroom)
df_summary_mushroom

,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Same Len Count
0,0,-6.56 ± 1.48,3.21 ± 1.03,0.17 ± 0.21,3.13 ± 0.81,-2.86 ± 2.13,93
1,1,7.12 ± 1.52,3.30 ± 1.71,0.62 ± 0.13,3.92 ± 1.76,1.61 ± 0.43,40


In [90]:
df_sonar = get_explain_results_n_increased("spambase", instances_to_explain)
df_summary_sonar = summarize_explanations_increased(df_sonar)
df_summary_sonar

,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Same Len Count
0,0,-3.70 ± 1.49,12.45 ± 2.37,0.48 ± 0.02,11.20 ± 2.47,0.37 ± 0.10,12
1,1,3.33 ± 1.47,13.25 ± 2.39,0.52 ± 0.02,15.66 ± 5.79,0.60 ± 0.11,10


# datasets thresholds

In [64]:
instances_to_explain = 100

In [65]:
df_saheart = get_explain_results_n("magic", instances_to_explain)
df_summary_saheart = summarize_explanations(df_saheart)
df_summary_saheart

0 same len on increase: 34
1 same len on increase: 13


,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Exp Len 25,MCT 25,Exp Len 50,MCT 50,Exp Len 75,MCT 75
0,0,0.16 ± 0.13,6.48 ± 1.01,0.44 ± 0.06,7.12 ± 1.03,0.38 ± 0.08,6.94 ± 0.96,0.32 ± 0.09,7.68 ± 0.89,0.25 ± 0.10,8.38 ± 0.75,0.18 ± 0.12
1,1,0.84 ± 0.15,3.52 ± 1.12,0.59 ± 0.09,4.43 ± 1.45,0.66 ± 0.10,4.03 ± 0.94,0.70 ± 0.11,4.38 ± 1.00,0.77 ± 0.14,5.29 ± 0.95,0.81 ± 0.15


In [66]:
df_adult = get_explain_results_n("adult", instances_to_explain)
df_summary_adult = summarize_explanations(df_adult)
df_summary_adult

0 same len on increase: 25
1 same len on increase: 12


,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Exp Len 25,MCT 25,Exp Len 50,MCT 50,Exp Len 75,MCT 75
0,0,0.23 ± 0.17,7.35 ± 2.46,0.45 ± 0.04,8.16 ± 2.65,0.40 ± 0.07,7.97 ± 2.19,0.36 ± 0.11,8.89 ± 1.94,0.30 ± 0.15,9.89 ± 1.68,0.26 ± 0.16
1,1,0.87 ± 0.14,4.86 ± 1.02,0.58 ± 0.07,5.40 ± 1.52,0.68 ± 0.11,5.06 ± 0.89,0.71 ± 0.11,5.60 ± 0.86,0.79 ± 0.13,6.69 ± 0.76,0.84 ± 0.14


In [67]:
df_mushroom = get_explain_results_n("mushroom", instances_to_explain)
df_summary_mushroom = summarize_explanations(df_mushroom)
df_summary_mushroom

0 same len on increase: 5
1 same len on increase: 1


,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Exp Len 25,MCT 25,Exp Len 50,MCT 50,Exp Len 75,MCT 75
0,0,0.01 ± 0.02,3.21 ± 1.03,0.17 ± 0.21,4.14 ± 0.85,0.08 ± 0.13,3.64 ± 1.25,0.06 ± 0.10,4.01 ± 1.80,0.03 ± 0.05,6.13 ± 1.86,0.01 ± 0.03
1,1,0.99 ± 0.03,3.30 ± 1.71,0.62 ± 0.13,4.29 ± 1.68,0.82 ± 0.10,4.65 ± 1.60,0.91 ± 0.08,6.03 ± 1.62,0.97 ± 0.05,8.03 ± 1.72,0.99 ± 0.04


In [68]:
df_sonar = get_explain_results_n("spambase", instances_to_explain)
df_summary_sonar = summarize_explanations(df_sonar)
df_summary_sonar

0 same len on increase: 16
1 same len on increase: 34


,class,Pred Margin,Exp Len Inicial,MCT Inicial,Exp Len increased,MCT increased,Exp Len 25,MCT 25,Exp Len 50,MCT 50,Exp Len 75,MCT 75
0,0,0.06 ± 0.08,12.45 ± 2.37,0.48 ± 0.02,13.01 ± 2.44,0.44 ± 0.04,14.78 ± 1.87,0.28 ± 0.08,16.74 ± 1.82,0.15 ± 0.09,18.47 ± 1.41,0.09 ± 0.09
1,1,0.92 ± 0.11,13.25 ± 2.39,0.52 ± 0.02,13.51 ± 2.36,0.55 ± 0.03,15.15 ± 1.81,0.71 ± 0.08,17.45 ± 1.45,0.83 ± 0.11,19.12 ± 1.44,0.89 ± 0.11
